# Future Sales Prediction - Kaggle Competition

### Modeling and Submission File

https://www.kaggle.com/competitions/competitive-data-science-predict-future-sales/

#### File descriptions
- sales_train.csv - the training set. Daily historical data from January 2013 to October 2015.

- test.csv - the test set. You need to forecast the sales for these shops and products for November 2015.

- sample_submission.csv - a sample submission file in the correct format.

- items.csv - supplemental information about the items/products.

- item_categories.csv  - supplemental information about the items categories.

- shops.csv- supplemental information about the shops.

#### Data fields

- ID - an Id that represents a (Shop, Item) tuple within the test set

- shop_id - unique identifier of a shop

- item_id - unique identifier of a product

- item_category_id - unique identifier of item category

- item_cnt_day - number of products sold. You are predicting a monthly amount of this measure

- item_price - current price of an item

- date - date in format dd/mm/yyyy

- date_block_num - a consecutive month number, used for convenience. January 2013 is 0, February 2013 is 1,..., October 2015 is 33

- item_name - name of item

- shop_name - name of shop

- item_category_name - name of item category

Import the necessary libraries and the final features dataset:

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
final_features = pd.read_csv('final_features.csv')

# Display all columns in the DataFrame, now that we have many features
pd.set_option('display.max_columns', None)

final_features.head()

,year,date_block_num,shop_id,item_category_id,item_id,item_price,item_cnt_month,revenue,med_price_shop,unique_items_in_shop,med_price_cat,unique_items_in_cat,med_price_all_items,weekends_in_month,holidays_in_month,spring,summer,fall,monthly_item_revenue,item_revenue_lagged_1m,item_revenue_lagged_3m,monthly_shop_revenue,shop_revenue_lagged_1m,shop_revenue_lagged_3m,monthly_cat_revenue,cat_revenue_lagged_1m,cat_revenue_lagged_3m,month_sin,month_cos
0,2013,0,2,2,27,2499.0,1.0,2499.0,399.0,728,1499.0,289,299.0,8,8,0,0,0,16275.0,0.0,0.0,1097686.8,0.0,0.0,2.216660e+07,0.0,0.0,0.5,0.866025
1,2013,0,2,2,1409,1398.5,1.0,1398.5,399.0,728,1499.0,289,299.0,8,8,0,0,0,41658.0,0.0,0.0,1097686.8,0.0,0.0,2.216660e+07,0.0,0.0,0.5,0.866025
2,2013,0,2,2,1467,899.0,1.0,899.0,399.0,728,1499.0,289,299.0,8,8,0,0,0,34895.0,0.0,0.0,1097686.8,0.0,0.0,2.216660e+07,0.0,0.0,0.5,0.866025
3,2013,0,2,2,1471,2599.0,2.0,5198.0,399.0,728,1499.0,289,299.0,8,8,0,0,0,746628.5,0.0,0.0,1097686.8,0.0,0.0,2.216660e+07,0.0,0.0,0.5,0.866025
4,2013,0,2,2,1832,1999.0,1.0,1999.0,399.0,728,1499.0,289,299.0,8,8,0,0,0,67965.0,0.0,0.0,1097686.8,0.0,0.0,2.216660e+07,0.0,0.0,0.5,0.866025


In [18]:
test = pd.read_csv('data/test.csv')
test.head()

,ID,shop_id,item_id
0,0,5,5037
1,1,5,5320
2,2,5,5233
3,3,5,5232
4,4,5,5268


### I. Declare training and validation sets

We are using 5-fold time series split cross-validation. However, because there are more than 1 observation in a single time frame (panel data), I have to split the folds manually.

Splitting plan (using date_block_num): There are 34 months in the training data (indexed from 0 to 33). I will split the folds so that for the smallest fold, there is at least sufficient data for a year to capture the cyclical characteristics of time series.

| Fold | Fold's training data          | Fold's validation data   |
|------|-------------------------------|--------------------------|
| 1    | ```date_block_num```: 0 -> 11 | ```date_block_num```: 12 |
| 2    | ```date_block_num```: 0 -> 17 | ```date_block_num```: 18 |
| 3    | ```date_block_num```: 0 -> 23 | ```date_block_num```: 24 |
| 4    | ```date_block_num```: 0 -> 29 | ```date_block_num```: 30 |
| 5    | ```date_block_num```: 0 -> 32 | ```date_block_num```: 33 |

In [4]:
# Split into the corresponding folds

# Fold 1
fold_1_train = final_features[final_features['date_block_num'].isin(range(12))]
fold_1_test = final_features[final_features['date_block_num'] == 12]

# Fold 2
fold_2_train = final_features[final_features['date_block_num'].isin(range(18))]
fold_2_test = final_features[final_features['date_block_num'] == 18]

# Fold 3
fold_3_train = final_features[final_features['date_block_num'].isin(range(24))]
fold_3_test = final_features[final_features['date_block_num'] == 24]

# Fold 4
fold_4_train = final_features[final_features['date_block_num'].isin(range(30))]
fold_4_test = final_features[final_features['date_block_num'] == 30]

# Fold 5
fold_5_train = final_features[final_features['date_block_num'].isin(range(33))]
fold_5_test = final_features[final_features['date_block_num'] == 33]

In [5]:
# Number of data points in each fold
print(f'''Fold 1: {fold_1_train.shape[0]} items in training set, {fold_1_test.shape[0]} items in validation set.
      Validation / Training = {fold_1_test.shape[0] / fold_1_train.shape[0]}''')
print('-------------------------------------------------------------')
print(f'''Fold 2: {fold_2_train.shape[0]} items in training set, {fold_2_test.shape[0]} items in validation set.
      Validation / Training = {fold_2_test.shape[0] / fold_2_train.shape[0]}''')
print('-------------------------------------------------------------')
print(f'''Fold 3: {fold_3_train.shape[0]} items in training set, {fold_3_test.shape[0]} items in validation set.
      Validation / Training = {fold_3_test.shape[0] / fold_3_train.shape[0]}''')
print('-------------------------------------------------------------')
print(f'''Fold 4: {fold_4_train.shape[0]} items in training set, {fold_4_test.shape[0]} items in validation set.
      Validation / Training = {fold_4_test.shape[0] / fold_4_train.shape[0]}''')
print('-------------------------------------------------------------')
print(f'''Fold 5: {fold_5_train.shape[0]} items in training set, {fold_5_test.shape[0]} items in validation set.
      Validation / Training = {fold_5_test.shape[0] / fold_5_train.shape[0]}''')

Fold 1: 687724 items in training set, 53320 items in validation set.
      Validation / Training = 0.07753110259348227
-------------------------------------------------------------
Fold 2: 974788 items in training set, 45694 items in validation set.
      Validation / Training = 0.04687583351456932
-------------------------------------------------------------
Fold 3: 1254511 items in training set, 46681 items in validation set.
      Validation / Training = 0.037210514694570235
-------------------------------------------------------------
Fold 4: 1480050 items in training set, 33527 items in validation set.
      Validation / Training = 0.022652613087395697
-------------------------------------------------------------
Fold 5: 1576741 items in training set, 31531 items in validation set.
      Validation / Training = 0.019997577281240228


In [6]:
# Split into features and targets

# Fold 1
X_train_1 = fold_1_train.drop('item_cnt_month', axis = 1).values
y_train_1 = fold_1_train[['item_cnt_month']].values.ravel()
X_test_1 = fold_1_test.drop('item_cnt_month', axis = 1).values
y_test_1 = fold_1_test[['item_cnt_month']].values.ravel()

# Fold 2
X_train_2 = fold_2_train.drop('item_cnt_month', axis = 1).values
y_train_2 = fold_2_train[['item_cnt_month']].values.ravel()
X_test_2 = fold_2_test.drop('item_cnt_month', axis = 1).values
y_test_2 = fold_2_test[['item_cnt_month']].values.ravel()

# Fold 3
X_train_3 = fold_3_train.drop('item_cnt_month', axis = 1).values
y_train_3 = fold_3_train[['item_cnt_month']].values.ravel()
X_test_3 = fold_3_test.drop('item_cnt_month', axis = 1).values
y_test_3 = fold_3_test[['item_cnt_month']].values.ravel()

# Fold 4
X_train_4 = fold_4_train.drop('item_cnt_month', axis = 1).values
y_train_4 = fold_4_train[['item_cnt_month']].values.ravel()
X_test_4 = fold_4_test.drop('item_cnt_month', axis = 1).values
y_test_4 = fold_4_test[['item_cnt_month']].values.ravel()

# Fold 5
X_train_5 = fold_5_train.drop('item_cnt_month', axis = 1).values
y_train_5 = fold_5_train[['item_cnt_month']].values.ravel()
X_test_5 = fold_5_test.drop('item_cnt_month', axis = 1).values
y_test_5 = fold_5_test[['item_cnt_month']].values.ravel()

In [7]:
# Check the shape of each X and y in training and test sets
print(f'''Fold 1: X_train: {X_train_1.shape}, y_train: {y_train_1.shape}, X_test: {X_test_1.shape}, y_test: {y_test_1.shape}''')
print(f'''Fold 2: X_train: {X_train_2.shape}, y_train: {y_train_2.shape}, X_test: {X_test_2.shape}, y_test: {y_test_2.shape}''')
print(f'''Fold 3: X_train: {X_train_3.shape}, y_train: {y_train_3.shape}, X_test: {X_test_3.shape}, y_test: {y_test_3.shape}''')
print(f'''Fold 4: X_train: {X_train_4.shape}, y_train: {y_train_4.shape}, X_test: {X_test_4.shape}, y_test: {y_test_4.shape}''')
print(f'''Fold 5: X_train: {X_train_5.shape}, y_train: {y_train_5.shape}, X_test: {X_test_5.shape}, y_test: {y_test_5.shape}''')

Fold 1: X_train: (687724, 28), y_train: (687724,), X_test: (53320, 28), y_test: (53320,)
Fold 2: X_train: (974788, 28), y_train: (974788,), X_test: (45694, 28), y_test: (45694,)
Fold 3: X_train: (1254511, 28), y_train: (1254511,), X_test: (46681, 28), y_test: (46681,)
Fold 4: X_train: (1480050, 28), y_train: (1480050,), X_test: (33527, 28), y_test: (33527,)
Fold 5: X_train: (1576741, 28), y_train: (1576741,), X_test: (31531, 28), y_test: (31531,)


In [8]:
# Example on fold 1
fold_1_train.tail()

,year,date_block_num,shop_id,item_category_id,item_id,item_price,item_cnt_month,revenue,med_price_shop,unique_items_in_shop,med_price_cat,unique_items_in_cat,med_price_all_items,weekends_in_month,holidays_in_month,spring,summer,fall,monthly_item_revenue,item_revenue_lagged_1m,item_revenue_lagged_3m,monthly_shop_revenue,shop_revenue_lagged_1m,shop_revenue_lagged_3m,monthly_cat_revenue,cat_revenue_lagged_1m,cat_revenue_lagged_3m,month_sin,month_cos
687719,2013,11,59,75,12727,1490.0,1.0,1490.0,349.0,1054,1290.0,98,398.0,9,0,0,0,0,28310.0,22275.5,34270.0,2514632.7,2.203782e+06,1.668340e+06,3.776741e+06,3.770479e+06,3393317.2,-2.449294e-16,1.0
687720,2013,11,59,83,22087,79.0,19.0,1501.0,349.0,1054,79.0,7,398.0,9,0,0,0,0,6803.0,5741.0,7172.0,2514632.7,2.203782e+06,1.668340e+06,2.369800e+04,1.682800e+04,23570.0,-2.449294e-16,1.0
687721,2013,11,59,83,22088,79.0,21.0,1659.0,349.0,1054,79.0,7,398.0,9,0,0,0,0,7391.0,6505.0,7418.0,2514632.7,2.203782e+06,1.668340e+06,2.369800e+04,1.682800e+04,23570.0,-2.449294e-16,1.0
687722,2013,11,59,83,22091,109.0,1.0,109.0,349.0,1054,79.0,7,398.0,9,0,0,0,0,3010.0,1618.0,4217.0,2514632.7,2.203782e+06,1.668340e+06,2.369800e+04,1.682800e+04,23570.0,-2.449294e-16,1.0
687723,2013,11,59,83,22092,109.0,7.0,763.0,349.0,1054,79.0,7,398.0,9,0,0,0,0,6322.0,2725.0,4763.0,2514632.7,2.203782e+06,1.668340e+06,2.369800e+04,1.682800e+04,23570.0,-2.449294e-16,1.0


In [9]:
# Set seed for reproducibility
my_seed = 275225

In [10]:
# What features can I apply scaling on (with MinMaxScaler)?

features_to_remove = ['shop_id', 'item_category_id', 'item_id', 'month_sin', 'month_cos', 'spring', 'summer', 'fall']
features_list = final_features.columns.tolist().copy() # start with all features that I have included

# Need to use an if statement so that this code can be rerun many times (this includes inplace removal of list elements)
for feature in features_to_remove:
    if feature in features_list:
        features_list.remove(feature)

print(features_list)

['year', 'date_block_num', 'item_price', 'item_cnt_month', 'revenue', 'med_price_shop', 'unique_items_in_shop', 'med_price_cat', 'unique_items_in_cat', 'med_price_all_items', 'weekends_in_month', 'holidays_in_month', 'monthly_item_revenue', 'item_revenue_lagged_1m', 'item_revenue_lagged_3m', 'monthly_shop_revenue', 'shop_revenue_lagged_1m', 'shop_revenue_lagged_3m', 'monthly_cat_revenue', 'cat_revenue_lagged_1m', 'cat_revenue_lagged_3m']


In [11]:
# https://stackoverflow.com/questions/22934609/get-indexes-of-multiple-pandas-columns-by-names
# Get index of the columns to pass to ColumnTransformer without error
# https://stackoverflow.com/questions/71715754/valueerror-specifying-the-columns-using-strings-is-only-supported-for-pandas-da

categorical_col_indices = [final_features.columns.get_loc(col) for col in ['shop_id', 'item_category_id', 'item_id']]
print(f'Indices of categorical columns: {categorical_col_indices}')

minmax_indices = [final_features.columns.get_loc(col) for col in features_list]
print(f'Indices of columns to apply MinMaxScaler on: {minmax_indices}')

Indices of categorical columns: [2, 3, 4]
Indices of columns to apply MinMaxScaler on: [0, 1, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 18, 19, 20, 21, 22, 23, 24, 25, 26]


In [12]:
from sklearn.preprocessing import TargetEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Encode categorical variables (shop_id, item_category_id, item_id) with TargetEncoder
# Encode all other variables (except for "month" and dummy variables) with MinMaxScaler, default range [0, 1]

target_enc = TargetEncoder(target_type = 'continuous', shuffle = True, random_state = my_seed)

transformer = ColumnTransformer(
    transformers = [('target_enc', target_enc, categorical_col_indices),
                    ('minmax_scaler', MinMaxScaler(), minmax_indices)],
    remainder = 'passthrough'
)

### V. Implementing regression models and assessing feature importance

In [13]:
from sklearn.metrics import root_mean_squared_error
import statistics
import time # I want to measure the efficiency of different machine learning models

# Custom function for training in many folds and many different models

folds = [[X_train_1, X_test_1, y_train_1, y_test_1], [X_train_2, X_test_2, y_train_2, y_test_2],
         [X_train_3, X_test_3, y_train_3, y_test_3], [X_train_4, X_test_4, y_train_4, y_test_4],
         [X_train_5, X_test_5, y_train_5, y_test_5]]

def kfold_regression(model_name, model, transformer_x, folds, output = 'df'):
    # Choices of output:
    # (1) 'df' (default, output a neatly formatted DataFrame of the results)
    # (2) 'score' (output only the mean RMSE)
    # (3) 'model' (output the model)

    rmse_folds = []

    pipe = Pipeline([('transformer', transformer_x), ('regression', model)])

    # I want to measure the time that Python fits the data and output the scores for all 5 folds
    start_time = time.perf_counter()

    # Each element in "folds" list should have the form [X_train, X_test, y_train, y_test]
    for fold in folds:
        pipe.fit(fold[0], fold[2])
        y_pred = pipe.predict(fold[1])
        rmse_folds.append(root_mean_squared_error(fold[3], y_pred))
    
    end_time = time.perf_counter()

    if output == 'df':
        df = pd.DataFrame({'model_name': [model_name], 'mean_rmse': [statistics.mean(rmse_folds)],
                       'std_rmse': [statistics.stdev(rmse_folds)], 'execution_time': [end_time - start_time]})
        return df
    elif output == 'score':
        return statistics.mean(rmse_folds)
    elif output == 'model':
        return pipe['regression']
    else:
        raise ValueError('The "output" parameter must be \'df\', \'score\', or \'model\'.')

In [14]:
# Import the necessary models
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

In [15]:
import re
import warnings
warnings.filterwarnings(action = 'ignore', message = r'^X does not have valid feature names')

In [16]:
kfold_regression('LightGBM (feature engineered)', LGBMRegressor(n_jobs = -1, random_state = my_seed, verbose = -1), transformer, folds)

,model_name,mean_rmse,std_rmse,execution_time
0,LightGBM (feature engineered),4.018975,3.878596,36.070046


In [ ]:
model_list = [('Linear Regression', LinearRegression(n_jobs = -1), transformer, folds),
               ('Lasso', Lasso(random_state = my_seed), transformer, folds),
               ('Ridge', Ridge(random_state = my_seed), transformer, folds),
               ('Elastic Net', ElasticNet(random_state = my_seed), transformer, folds)]

overall_results = pd.concat([kfold_regression(model_name, model, transformer, folds) for model_name, model, transformer, folds in model_list], axis = 0, ignore_index = True)
overall_results.sort_values(['mean_rmse', 'std_rmse', 'execution_time'], inplace = True, ignore_index = True)
overall_results

,model_name,mean_rmse,std_rmse,execution_time
0,Elastic Net,1.995931,2.520029,12.761016
1,Lasso,1.995931,2.520029,38.955910
2,Ridge,2.093282,2.446418,19.947494
3,Linear Regression,2.093299,2.446407,36.233729


In [ ]:
# WARNING: THIS CODE BLOCK WILL TAKE A LONG TIME TO RUN (est. > 40 mins)

model_list_2 = [('Decision Tree', DecisionTreeRegressor(random_state = my_seed), transformer, folds),
              ('Random Forest', RandomForestRegressor(n_jobs = -1, random_state = my_seed), transformer, folds),
              ('AdaBoost', AdaBoostRegressor(random_state = my_seed), transformer, folds),
              ('Gradient Boosting', GradientBoostingRegressor(random_state = my_seed), transformer, folds),
              ('Hist-based Gradient Boosting', HistGradientBoostingRegressor(random_state = my_seed), transformer, folds),
              ('XGBoost', XGBRegressor(n_jobs = -1, random_state = my_seed), transformer, folds),
              ('CatBoost', CatBoostRegressor(random_state = my_seed, verbose = False), transformer, folds),
              ('LightGBM', LGBMRegressor(n_jobs = -1, random_state = my_seed, verbose = -1), transformer, folds)]

overall_results = pd.concat([kfold_regression(model_name, model, transformer, folds) for model_name, model, transformer, folds in model_list_2], axis = 0, ignore_index = True)
overall_results.sort_values(['mean_rmse', 'std_rmse', 'execution_time'], inplace = True, ignore_index = True)
overall_results

,model_name,mean_rmse,std_rmse,execution_time
0,Gradient Boosting,1.314370,1.584073,995.316409
1,Random Forest,1.414142,1.761584,529.898067
2,LightGBM,1.469164,1.988627,16.464381
3,CatBoost,1.647621,1.750774,217.158482
4,Decision Tree,1.683901,1.632204,182.598874
5,AdaBoost,1.727030,1.885637,432.293311
6,Hist-based Gradient Boosting,1.902756,2.348932,20.536378
7,XGBoost,2.738961,2.255959,17.657261


### II. Assessing feature importance

In [108]:
def coef_or_feature_importances(model, folds, output = 'df'):
    # output takes 2 possible values: 'df' (pandas DataFrame) or 'np' (NumPy array)

    importances_folds = []

    pipe = Pipeline([('transformer', transformer), ('regression', model)])

    # Each element in "folds" list should have the form [X_train, X_test, y_train, y_test]
    for fold in folds:
        pipe.fit(fold[0], fold[2])

        try:
            importances_folds.append(pipe['regression'].coef_)

        except AttributeError: # for tree-based models which do not have coefficients attribute
            importances_folds.append(pipe['regression'].feature_importances_)
    
    if (output == 'df') or (output == 'np'):
        importances_folds = np.array(importances_folds)
        if output == 'df':
            importances_folds = pd.DataFrame(importances_folds)
            # Transpose the DataFrame so that rows = features, columns = folds
            importances_folds = importances_folds.transpose()
            # Give the features a name
            importances_folds.index = fold_1_train.drop('item_cnt_month', axis = 1).columns
            importances_folds['mean_coef_or_importance'] = importances_folds.mean(axis = 1)
            importances_folds.sort_values('mean_coef_or_importance', ascending = False, inplace = True)

    return importances_folds

#### Linear-based models:

In [ ]:
# Mean absolute coefficient across 5 folds of Linear Regression (top 7 only)
linreg_coef = coef_or_feature_importances(LinearRegression(n_jobs = -1), folds)
linreg_coef['abs_mean_coef'] = linreg_coef['mean_coef_or_importance'].abs()
linreg_coef.sort_values('abs_mean_coef', ascending = False, inplace = True)
linreg_coef.iloc[:7]

,0,1,2,3,4,mean_coef_or_importance,abs_mean_coef
med_price_shop,0.289062,0.364065,0.616721,4.044620,3.885479,1.839989,1.839989
shop_id,1.212450,1.158344,1.110352,1.037005,1.037697,1.111170,1.111170
item_price,1.224198,1.587866,1.238001,0.626277,0.536267,1.042522,1.042522
unique_items_in_shop,0.927807,0.827209,0.808218,0.815631,0.803608,0.836494,0.836494
holidays_in_month,0.448222,0.524611,0.357545,0.325824,0.315205,0.394281,0.394281
item_id,-0.282016,-0.168054,-0.468775,-0.450819,-0.563629,-0.386659,0.386659
date_block_num,0.209291,0.272792,0.326647,0.375655,0.389165,0.314710,0.314710


In [118]:
# Mean absolute coefficient across 5 folds of Ridge Regression (top 7 only)
ridge_coef = coef_or_feature_importances(Ridge(random_state = my_seed), folds)
ridge_coef['abs_mean_coef'] = ridge_coef['mean_coef_or_importance'].abs()
ridge_coef.sort_values('abs_mean_coef', ascending = False, inplace = True)
ridge_coef.iloc[:7]

,0,1,2,3,4,mean_coef_or_importance,abs_mean_coef
med_price_shop,0.289467,0.364502,0.616854,3.993236,3.840205,1.820853,1.820853
shop_id,1.212453,1.158330,1.110327,1.037005,1.037699,1.111163,1.111163
unique_items_in_shop,0.927765,0.827205,0.808174,0.814444,0.802537,0.836025,0.836025
item_price,0.887063,1.224976,1.007409,0.533210,0.462489,0.823030,0.823030
holidays_in_month,0.448198,0.524570,0.357554,0.326010,0.315362,0.394339,0.394339
item_id,-0.282019,-0.168098,-0.468668,-0.450213,-0.563047,-0.386409,0.386409
date_block_num,0.209247,0.272762,0.326631,0.375755,0.389255,0.314730,0.314730


I couldn't calculate feature importance (through coefficient) with Lasso and ElasticNet regression because the models zeroed out all coefficients, making comparison very difficult.

#### Tree-based models:

In [115]:
# Mean feature importances across 5 folds of LightGBM (top 7 only)
lgbm_importances = coef_or_feature_importances(LGBMRegressor(n_jobs = -1, random_state = my_seed, verbose = -1), folds)
lgbm_importances.iloc[:7]

,0,1,2,3,4,mean_coef_or_importance
shop_id,576,473,427,405,372,450.6
unique_items_in_shop,460,434,406,408,491,439.8
year,411,408,410,441,514,436.8
date_block_num,445,356,360,383,334,375.6
item_price,330,370,402,364,354,364.0
unique_items_in_cat,219,241,299,258,274,258.2
med_price_shop,164,159,215,298,235,214.2


In [116]:
# Mean feature importances across 5 folds of XGBoost (top 7 only)
xgb_importances = coef_or_feature_importances(XGBRegressor(n_jobs = -1, random_state = my_seed), folds)
xgb_importances.iloc[:7]

,0,1,2,3,4,mean_coef_or_importance
item_id,0.294833,0.220563,0.224927,0.395025,0.047091,0.236488
med_price_shop,0.040224,0.037634,0.094507,0.288312,0.309881,0.154112
unique_items_in_shop,0.216597,0.139927,0.091294,0.045087,0.077975,0.114176
shop_id,0.140726,0.157355,0.085660,0.056715,0.066453,0.101382
med_price_all_items,0.000000,0.120440,0.120051,0.001149,0.199962,0.088321
date_block_num,0.082870,0.078359,0.071445,0.043447,0.016105,0.058445
year,0.033873,0.058669,0.025845,0.070213,0.058631,0.049446


CatBoost takes a very long time to run my feature importances function, so I will not include it here.

Looking at the top 7 "most important features" output by different models, I will choose the following features: ```med_price_shop```, ```shop_id```, ```unique_items_in_shop```, ```item_price```.

_(If I have more time, I will develop a more systematic approach for feature selection, but this is what I'm doing at the moment)_

In [132]:
# Edit the features for each fold
fold_1_train_new = fold_1_train.loc[:, ['med_price_shop', 'shop_id', 'unique_items_in_shop', 'item_price', 'item_cnt_month']]
fold_1_test_new = fold_1_test.loc[:, ['med_price_shop', 'shop_id', 'unique_items_in_shop', 'item_price', 'item_cnt_month']]

fold_2_train_new = fold_2_train.loc[:, ['med_price_shop', 'shop_id', 'unique_items_in_shop', 'item_price', 'item_cnt_month']]
fold_2_test_new = fold_2_test.loc[:, ['med_price_shop', 'shop_id', 'unique_items_in_shop', 'item_price', 'item_cnt_month']]

fold_3_train_new = fold_3_train.loc[:, ['med_price_shop', 'shop_id', 'unique_items_in_shop', 'item_price', 'item_cnt_month']]
fold_3_test_new = fold_3_test.loc[:, ['med_price_shop', 'shop_id', 'unique_items_in_shop', 'item_price', 'item_cnt_month']]

fold_4_train_new = fold_4_train.loc[:, ['med_price_shop', 'shop_id', 'unique_items_in_shop', 'item_price', 'item_cnt_month']]
fold_4_test_new = fold_4_test.loc[:, ['med_price_shop', 'shop_id', 'unique_items_in_shop', 'item_price', 'item_cnt_month']]

fold_5_train_new = fold_5_train.loc[:, ['med_price_shop', 'shop_id', 'unique_items_in_shop', 'item_price', 'item_cnt_month']]
fold_5_test_new = fold_5_test.loc[:, ['med_price_shop', 'shop_id', 'unique_items_in_shop', 'item_price', 'item_cnt_month']]

In [138]:
# As an example
fold_5_train_new.head()

,med_price_shop,shop_id,unique_items_in_shop,item_price,item_cnt_month
0,196.0,0,2385,1322.0,10.0
1,196.0,0,2385,560.0,1.0
2,196.0,0,2385,806.0,4.0
3,196.0,0,2385,2231.0,5.0
4,196.0,0,2385,2381.0,1.0


In [133]:
# Split into features and targets again

# Fold 1
X_train_1_new = fold_1_train_new.drop('item_cnt_month', axis = 1).values
y_train_1_new = fold_1_train_new[['item_cnt_month']].values.ravel()
X_test_1_new = fold_1_test_new.drop('item_cnt_month', axis = 1).values
y_test_1_new = fold_1_test_new[['item_cnt_month']].values.ravel()

# Fold 2
X_train_2_new = fold_2_train_new.drop('item_cnt_month', axis = 1).values
y_train_2_new = fold_2_train_new[['item_cnt_month']].values.ravel()
X_test_2_new = fold_2_test_new.drop('item_cnt_month', axis = 1).values
y_test_2_new = fold_2_test_new[['item_cnt_month']].values.ravel()

# Fold 3
X_train_3_new = fold_3_train_new.drop('item_cnt_month', axis = 1).values
y_train_3_new = fold_3_train_new[['item_cnt_month']].values.ravel()
X_test_3_new = fold_3_test_new.drop('item_cnt_month', axis = 1).values
y_test_3_new = fold_3_test_new[['item_cnt_month']].values.ravel()

# Fold 4
X_train_4_new = fold_4_train_new.drop('item_cnt_month', axis = 1).values
y_train_4_new = fold_4_train_new[['item_cnt_month']].values.ravel()
X_test_4_new = fold_4_test_new.drop('item_cnt_month', axis = 1).values
y_test_4_new = fold_4_test_new[['item_cnt_month']].values.ravel()

# Fold 5
X_train_5_new = fold_5_train_new.drop('item_cnt_month', axis = 1).values
y_train_5_new = fold_5_train_new[['item_cnt_month']].values.ravel()
X_test_5_new = fold_5_test_new.drop('item_cnt_month', axis = 1).values
y_test_5_new = fold_5_test_new[['item_cnt_month']].values.ravel()

In [143]:
folds_new = [[X_train_1_new, X_test_1_new, y_train_1_new, y_test_1_new], [X_train_2_new, X_test_2_new, y_train_2_new, y_test_2_new],
             [X_train_3_new, X_test_3_new, y_train_3_new, y_test_3_new], [X_train_4_new, X_test_4_new, y_train_4_new, y_test_4_new],
             [X_train_5_new, X_test_5_new, y_train_5_new, y_test_5_new]]

# Remember to change the transformer accordingly with the new features
transformer_new = ColumnTransformer(
    transformers = [('target_enc', target_enc, [1]),
                    ('minmax_scaler', MinMaxScaler(), [0, 2, 3])],
    remainder = 'passthrough'
)

In [ ]:
model_list = [('Linear Regression', LinearRegression(n_jobs = -1), transformer_new, folds_new),
               ('Lasso', Lasso(random_state = my_seed), transformer_new, folds_new),
               ('Ridge', Ridge(random_state = my_seed), transformer_new, folds_new),
               ('Elastic Net', ElasticNet(random_state = my_seed), transformer_new, folds_new)]

overall_results = pd.concat([kfold_regression(model_name, model, transformer, folds) for model_name, model, transformer, folds in model_list], axis = 0, ignore_index = True)
overall_results.sort_values(['mean_rmse', 'std_rmse', 'execution_time'], inplace = True, ignore_index = True)
overall_results

,model_name,mean_rmse,std_rmse,execution_time
0,Elastic Net,1.995931,2.520029,1.539344
1,Lasso,1.995931,2.520029,1.644030
2,Ridge,2.003311,2.513683,1.508836
3,Linear Regression,2.003394,2.513621,1.840203


In [ ]:
# WARNING: THIS CODE BLOCK WILL TAKE A LONG TIME TO RUN (est. 30 mins)

model_list_2 = [('Decision Tree', DecisionTreeRegressor(random_state = my_seed), transformer_new, folds_new),
              ('Random Forest', RandomForestRegressor(n_jobs = -1, random_state = my_seed), transformer_new, folds_new),
              ('AdaBoost', AdaBoostRegressor(random_state = my_seed), transformer_new, folds_new),
              ('Gradient Boosting', GradientBoostingRegressor(random_state = my_seed), transformer_new, folds_new),
              ('Hist-based Gradient Boosting', HistGradientBoostingRegressor(random_state = my_seed), transformer_new, folds_new),
              ('XGBoost', XGBRegressor(n_jobs = -1, random_state = my_seed), transformer_new, folds_new),
              ('CatBoost', CatBoostRegressor(random_state = my_seed, verbose = False), transformer_new, folds_new),
              ('LightGBM', LGBMRegressor(n_jobs = -1, random_state = my_seed, verbose = -1), transformer_new, folds_new)]

overall_results = pd.concat([kfold_regression(model_name, model, transformer, folds) for model_name, model, transformer, folds in model_list_2], axis = 0, ignore_index = True)
overall_results.sort_values(['mean_rmse', 'std_rmse', 'execution_time'], inplace = True, ignore_index = True)
overall_results

,model_name,mean_rmse,std_rmse,execution_time
0,AdaBoost,1.116967,1.067410,218.973728
1,Gradient Boosting,1.244388,1.247079,579.035178
2,Random Forest,1.247933,1.686888,102.238504
3,LightGBM,1.938657,2.309566,22.255609
4,CatBoost,1.964565,1.797574,743.932278
5,Hist-based Gradient Boosting,1.966352,2.479180,27.655547
6,XGBoost,2.142089,2.394970,24.997484
7,Decision Tree,2.495480,2.137287,17.764580


### III. Hyperparameter tuning

In [49]:
import optuna
from optuna.samplers import TPESampler

In [55]:
# Tuning for ElasticNet
def objective_elasticnet(trial):
    alpha = trial.suggest_float('alpha', 10**(-3), 10**2, log = True)
    l1_ratio = trial.suggest_float('l1_ratio', 0.05, 0.95, step = 0.05)
    tol = trial.suggest_float('tol', 10**(-5), 10**(-2), log = True)

    # Increase max_iter to 10000 for better results
    elasticnet_tuned = ElasticNet(random_state = my_seed, max_iter = 10000, alpha = alpha,
                                  l1_ratio = l1_ratio, tol = tol)

    try:
        score = kfold_regression('Elastic Net', elasticnet_tuned, folds, output = 'score')

        if trial.should_prune():
            raise optuna.TrialPruned()
    
        return score
    
    except Exception:
        pass

    # Use the try-except syntax to make sure the study does not end when a trial fails 

In [ ]:
# WARNING: THIS CODE BLOCK WILL TAKE A LONG TIME TO RUN (est. 2 hours)

# https://optuna.readthedocs.io/en/stable/reference/generated/optuna.pruners.MedianPruner.html
# Use MedianPruner as a form of early stopping
study1 = optuna.create_study(sampler = TPESampler(seed = my_seed), direction = 'minimize',
    pruner = optuna.pruners.MedianPruner(n_startup_trials = 3, n_warmup_steps = 1, interval_steps = 1, n_min_trials = 1))

# Not printing out Optuna updates periodically, including failed trials
optuna.logging.set_verbosity(optuna.logging.ERROR)
study1.optimize(objective_elasticnet, n_trials = 500)

# https://optuna.readthedocs.io/en/stable/reference/generated/optuna.trial.FrozenTrial.html
elasticnet_best_params = study1.best_trial.params

print(f'Trial number {study1.best_trial.number} has the best performance, with a score of {study1.best_trial.value}.')
print(f'The best parameters are: {study1.best_trial.params}')

[I 2025-08-07 10:43:25,233] A new study created in memory with name: no-name-75c7ddf2-97e8-4ce2-b8de-debafa0fdb06


Trial number 264 has the best performance, with a score of 1.995575197368847.
The best parameters are: {'alpha': 0.3668848655571989, 'l1_ratio': 0.35000000000000003, 'tol': 0.0027079448878869965}


In [59]:
# Tuning for LightGBM
# https://medium.com/@amitsinghrajput_92567/understanding-hyperparameters-in-decision-trees-xgboost-and-lightgbm-7b64cfed77f0

def objective_lgbm(trial):
    num_leaves = trial.suggest_int('num_leaves', 5, 100, step = 1)
    max_depth = trial.suggest_int('max_depth', 5, 100, step = 1)
    learning_rate = trial.suggest_float('learning_rate', 10**(-4), 10**(-1), log = True)
    min_split_gain = trial.suggest_float('min_split_gain', 0.05, 0.95, step = 0.05)
    min_child_weight = trial.suggest_float('min_child_weight', 10**(-4), 10**1, log = True)
    min_child_samples = trial.suggest_int('min_child_samples', 10, 10000, log = True)
    subsample = trial.suggest_float('subsample', 0.05, 0.95, step = 0.05)
    subsample_freq = trial.suggest_int('subsample_freq', 10, 1000, log = True)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.05, 0.95, step = 0.05)
    reg_alpha = trial.suggest_float('reg_alpha', 10**(-3), 10**1, log = True)
    reg_lambda = trial.suggest_float('reg_lambda', 10**(-3), 10**1, log = True)

    # Set n_estimators = 10000, random_state, verbose, and n_jobs to preferred value
    lgbm_tuned = LGBMRegressor(n_estimators = 10000, random_state = my_seed, n_jobs = -1, verbose = -1,
                               num_leaves = num_leaves, max_depth = max_depth, learning_rate = learning_rate,
                               min_split_gain = min_split_gain, min_child_weight = min_child_weight,
                               min_child_samples = min_child_samples, subsample = subsample,
                               subsample_freq = subsample_freq, colsample_bytree = colsample_bytree,
                               reg_alpha = reg_alpha, reg_lambda = reg_lambda)

    try:
        score = kfold_regression('LightGBM', lgbm_tuned, folds, output = 'score')

        if trial.should_prune():
            raise optuna.TrialPruned()
    
        return score
    
    except Exception:
        pass

    # Use the try-except syntax to make sure the study does not end when a trial fails

In [62]:
# WARNING: THIS CODE BLOCK WILL TAKE A LONG TIME TO RUN

study = optuna.create_study(sampler = TPESampler(seed = my_seed), direction = 'minimize',
    pruner = optuna.pruners.MedianPruner(n_startup_trials = 3, n_warmup_steps = 1, interval_steps = 1, n_min_trials = 1))

# optuna.logging.set_verbosity(optuna.logging.ERROR) # can uncomment this command to avoid excessive logging by Optuna
study.optimize(objective_lgbm, n_trials = 10)

lgbm_best_params = study.best_trial.params

print(f'Trial number {study.best_trial.number} has the best performance, with a score of {study.best_trial.value}.')
print(f'The best parameters are: {study.best_trial.params}')

Trial number 0 has the best performance, with a score of 1.5233667675060847.
The best parameters are: {'num_leaves': 31, 'max_depth': 64, 'learning_rate': 0.023251443377156827, 'min_split_gain': 0.6000000000000001, 'min_child_weight': 0.053160726817981174, 'min_child_samples': 64, 'subsample': 0.8, 'subsample_freq': 189, 'colsample_bytree': 0.95, 'reg_alpha': 0.31549086474120447, 'reg_lambda': 0.08921894140014044}


### IV. Prepare data for final submission

In [48]:
# Declare final model for submission
from sklearn.ensemble import StackingRegressor

#final_model = StackingRegressor(estimators = [('elasticnet', ElasticNet(random_state = my_seed, max_iter = 10000, **elasticnet_best_params)),
                                              #('lightgbm', LGBMRegressor(n_estimators = 10000, random_state = my_seed, n_jobs = -1, verbose = -1, **lgbm_best_params))],
                                #n_jobs = -1)

In [49]:
test_shop_items = test[['shop_id', 'item_id']].copy()
test_shop_items.head() # this is the test data stripped of its ID

,shop_id,item_id
0,5,5037
1,5,5320
2,5,5233
3,5,5232
4,5,5268


In [50]:
# Set constant data for some columns
test_features = test_shop_items.copy()

test_features['year'] = 2013
test_features['date_block_num'] = 34 # number of months elapsed since Jan 2013

test_features['spring'] = 0; test_features['summer'] = 0; test_features['fall'] = 1 # November is in the fall season
test_features['weekends_in_month'] = 9
test_features['holidays_in_month'] = 1 # Russia's Unity Day in 4 November

# For the month_sin and month_cos trigonometric encoding, replace the month number with 11
test_features['month_sin'] = np.sin(11 / 12 * 2 * np.pi)
test_features['month_cos'] = np.cos(11 / 12 * 2 * np.pi)

test_features.head()

,shop_id,item_id,year,date_block_num,spring,summer,fall,weekends_in_month,holidays_in_month,month_sin,month_cos
0,5,5037,2013,34,0,0,1,9,1,-0.5,0.866025
1,5,5320,2013,34,0,0,1,9,1,-0.5,0.866025
2,5,5233,2013,34,0,0,1,9,1,-0.5,0.866025
3,5,5232,2013,34,0,0,1,9,1,-0.5,0.866025
4,5,5268,2013,34,0,0,1,9,1,-0.5,0.866025


In [51]:
# Retrieve item category and item price (each item only belongs to 1 category and has 1 price)
item_info = final_features[['shop_id', 'item_id', 'item_category_id', 'item_price']].copy()

# Keep only the last occurence of (shop_id, item_id) pair (showing the latest price)
item_info.drop_duplicates(subset = ['shop_id', 'item_id'], keep = 'last', inplace = True, ignore_index = True)
item_info.head()

,shop_id,item_id,item_category_id,item_price
0,0,5572,2,1322.0
1,0,5575,2,806.0
2,0,5576,2,2231.0
3,0,5612,2,3623.0
4,0,5623,2,294.0


In [52]:
# Merge item information to the test set
item_info = test_shop_items.merge(item_info, on = ['shop_id', 'item_id'], how = 'left')
item_info.drop_duplicates(inplace = True, ignore_index = True)
item_info.head()

,shop_id,item_id,item_category_id,item_price
0,5,5037,19.0,749.5
1,5,5320,NaN,NaN
2,5,5233,19.0,1199.0
3,5,5232,23.0,599.0
4,5,5268,NaN,NaN


In [53]:
item_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 214200 entries, 0 to 214199
Data columns (total 4 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   shop_id           214200 non-null  int64  
 1   item_id           214200 non-null  int64  
 2   item_category_id  104268 non-null  float64
 3   item_price        104268 non-null  float64
dtypes: float64(2), int64(2)
memory usage: 6.5 MB


In [54]:
# Create a new item category ID that denotes "unknown"
new_cat_id = item_categories['item_category_id'].max() + 1 # new identifier for unknown category
print(new_cat_id)

84


In [55]:
# item_info['item_category_id']
item_info['item_category_id'] = item_info['item_category_id'].fillna(new_cat_id)
item_info.head()

,shop_id,item_id,item_category_id,item_price
0,5,5037,19.0,749.5
1,5,5320,84.0,NaN
2,5,5233,19.0,1199.0
3,5,5232,23.0,599.0
4,5,5268,84.0,NaN


In [56]:
from sklearn.impute import SimpleImputer
imp = SimpleImputer(strategy = 'median')

imputed_data = imp.fit_transform(item_info) # outputs a NumPy array
new_item_info = pd.DataFrame(imputed_data, columns = item_info.columns) # convert into a DataFrame
new_item_info.head()

,shop_id,item_id,item_category_id,item_price
0,5.0,5037.0,19.0,749.5
1,5.0,5320.0,84.0,449.0
2,5.0,5233.0,19.0,1199.0
3,5.0,5232.0,23.0,599.0
4,5.0,5268.0,84.0,449.0


In [57]:
# Create new features based on imputed data (for shops)
new_item_info_shop = new_item_info[['shop_id', 'item_id', 'item_price']].copy()
new_item_info_shop = new_item_info_shop.groupby('shop_id', as_index = False).agg({'item_price': 'median', 'item_id': 'nunique'})
new_item_info_shop.rename(columns = {'item_price': 'med_price_shop', 'item_id': 'unique_items_in_shop'}, inplace = True)
new_item_info_shop.head()

,shop_id,med_price_shop,unique_items_in_shop
0,2.0,449.0,5100
1,3.0,449.0,5100
2,4.0,449.0,5100
3,5.0,449.0,5100
4,6.0,449.0,5100


In [58]:
# Create new features based on imputed data (for categories)
new_item_info_cat = new_item_info[['item_category_id', 'item_id', 'item_price']].copy()
new_item_info_cat = new_item_info_cat.groupby('item_category_id', as_index = False).agg({'item_price': 'median', 'item_id': 'nunique'})
new_item_info_cat.rename(columns = {'item_price': 'med_price_cat', 'item_id': 'unique_items_in_cat'}, inplace = True)
new_item_info_cat.head()

,item_category_id,med_price_cat,unique_items_in_cat
0,2.0,2990.0,10
1,3.0,1390.0,27
2,5.0,599.0,4
3,6.0,2290.0,9
4,7.0,2489.5,23


In [59]:
# Get median price of all items (including imputed) sold in November 2015 - test data 
new_item_info['med_price_all_items'] = new_item_info['item_price'].median()
new_item_info.head()

,shop_id,item_id,item_category_id,item_price,med_price_all_items
0,5.0,5037.0,19.0,749.5,449.0
1,5.0,5320.0,84.0,449.0,449.0
2,5.0,5233.0,19.0,1199.0,449.0
3,5.0,5232.0,23.0,599.0,449.0
4,5.0,5268.0,84.0,449.0,449.0


In [60]:
# Merge into the existing new_item_info DataFrame
item_info = new_item_info.merge(new_item_info_shop, how = 'left', on = 'shop_id')\
    .merge(new_item_info_cat, how = 'left', on = 'item_category_id')

item_info.head()

,shop_id,item_id,item_category_id,item_price,med_price_all_items,med_price_shop,unique_items_in_shop,med_price_cat,unique_items_in_cat
0,5.0,5037.0,19.0,749.5,449.0,449.0,5100,1199.0,137
1,5.0,5320.0,84.0,449.0,449.0,449.0,5100,449.0,5100
2,5.0,5233.0,19.0,1199.0,449.0,449.0,5100,1199.0,137
3,5.0,5232.0,23.0,599.0,449.0,449.0,5100,1199.0,143
4,5.0,5268.0,84.0,449.0,449.0,449.0,5100,449.0,5100


In [61]:
# Now merge to create the full test set
test_features = test_features.merge(item_info, how = 'left', on = ['shop_id', 'item_id'])
test_features.head()

,shop_id,item_id,year,date_block_num,spring,summer,fall,weekends_in_month,holidays_in_month,month_sin,month_cos,item_category_id,item_price,med_price_all_items,med_price_shop,unique_items_in_shop,med_price_cat,unique_items_in_cat
0,5,5037,2013,34,0,0,1,9,1,-0.5,0.866025,19.0,749.5,449.0,449.0,5100,1199.0,137
1,5,5320,2013,34,0,0,1,9,1,-0.5,0.866025,84.0,449.0,449.0,449.0,5100,449.0,5100
2,5,5233,2013,34,0,0,1,9,1,-0.5,0.866025,19.0,1199.0,449.0,449.0,5100,1199.0,137
3,5,5232,2013,34,0,0,1,9,1,-0.5,0.866025,23.0,599.0,449.0,449.0,5100,1199.0,143
4,5,5268,2013,34,0,0,1,9,1,-0.5,0.866025,84.0,449.0,449.0,449.0,5100,449.0,5100


In [62]:
# Create submission training and test data
submission_train = final_features.copy()

# After checking data types (truncated), I need to change data type of the weekends_in_month and holidays_in_month columns from object to int
submission_train['weekends_in_month'] = submission_train['weekends_in_month'].astype('int')
submission_train['holidays_in_month'] = submission_train['weekends_in_month'].astype('int')

submission_train.head()

,year,date_block_num,shop_id,item_category_id,item_id,item_price,item_cnt_month,med_price_shop,unique_items_in_shop,med_price_cat,unique_items_in_cat,med_price_all_items,weekends_in_month,holidays_in_month,spring,summer,fall,month_sin,month_cos
0,2013,0,0,2,5572,1322.0,10.0,196.0,2385,2390.0,37,299.0,8,8,0,0,0,0.5,0.866025
1,2013,0,0,2,5573,560.0,1.0,196.0,2385,2390.0,37,299.0,8,8,0,0,0,0.5,0.866025
2,2013,0,0,2,5575,806.0,4.0,196.0,2385,2390.0,37,299.0,8,8,0,0,0,0.5,0.866025
3,2013,0,0,2,5576,2231.0,5.0,196.0,2385,2390.0,37,299.0,8,8,0,0,0,0.5,0.866025
4,2013,0,0,2,5609,2381.0,1.0,196.0,2385,2390.0,37,299.0,8,8,0,0,0,0.5,0.866025


In [63]:
# Reorder the columns to align with the training data
submission_test = test_features.loc[:, final_features.drop('item_cnt_month', axis = 1).columns.tolist()]
submission_test.head()

,year,date_block_num,shop_id,item_category_id,item_id,item_price,med_price_shop,unique_items_in_shop,med_price_cat,unique_items_in_cat,med_price_all_items,weekends_in_month,holidays_in_month,spring,summer,fall,month_sin,month_cos
0,2013,34,5,19.0,5037,749.5,449.0,5100,1199.0,137,449.0,9,1,0,0,1,-0.5,0.866025
1,2013,34,5,84.0,5320,449.0,449.0,5100,449.0,5100,449.0,9,1,0,0,1,-0.5,0.866025
2,2013,34,5,19.0,5233,1199.0,449.0,5100,1199.0,137,449.0,9,1,0,0,1,-0.5,0.866025
3,2013,34,5,23.0,5232,599.0,449.0,5100,1199.0,143,449.0,9,1,0,0,1,-0.5,0.866025
4,2013,34,5,84.0,5268,449.0,449.0,5100,449.0,5100,449.0,9,1,0,0,1,-0.5,0.866025


Do the train-test split and implementing a pipeline process to submit.

In [80]:
# Full pipeline
X_train_submission = submission_train.drop('item_cnt_month', axis = 1).values
y_train_submission = submission_train[['item_cnt_month']].values.ravel()
X_test_submission = submission_test.values

full_pipe = Pipeline([('target_enc', transformer), ('regression', final_model)])

full_pipe.fit(X_train_submission, y_train_submission)

submission_test['item_cnt_month_unadjusted'] = full_pipe.predict(X_test_submission)

# Cap the predicted values in the [0, 20] range
submission_test['item_cnt_month'] = submission_test['item_cnt_month_unadjusted'].case_when\
    ([(submission_test['item_cnt_month_unadjusted'] < 0, 0),
      (submission_test['item_cnt_month_unadjusted'] > 20, 20),
      (np.logical_and(submission_test['item_cnt_month_unadjusted'] >= 0, submission_test['item_cnt_month_unadjusted'] <= 20),
       submission_test['item_cnt_month_unadjusted'])])

submission_test.head()

,year,date_block_num,shop_id,item_category_id,item_id,item_price,med_price_shop,unique_items_in_shop,med_price_cat,unique_items_in_cat,med_price_all_items,weekends_in_month,holidays_in_month,spring,summer,fall,month_sin,month_cos,item_cnt_month_unadjusted,item_cnt_month
0,2013,34,5,19.0,5037,749.5,449.0,5100,1199.0,137,449.0,9,1,0,0,1,-0.5,0.866025,1.118458,1.118458
1,2013,34,5,84.0,5320,449.0,449.0,5100,449.0,5100,449.0,9,1,0,0,1,-0.5,0.866025,1.414916,1.414916
2,2013,34,5,19.0,5233,1199.0,449.0,5100,1199.0,137,449.0,9,1,0,0,1,-0.5,0.866025,1.593280,1.593280
3,2013,34,5,23.0,5232,599.0,449.0,5100,1199.0,143,449.0,9,1,0,0,1,-0.5,0.866025,2.004942,2.004942
4,2013,34,5,84.0,5268,449.0,449.0,5100,449.0,5100,449.0,9,1,0,0,1,-0.5,0.866025,1.414916,1.414916


In [81]:
# Extract only the necessary columns
submission_without_id = submission_test[['shop_id', 'item_id', 'item_cnt_month']].copy()

# Merge the ID column, then remove the shop_id and item_id column to fit with sample submission
submission = submission_without_id.merge(test, on = ['shop_id', 'item_id'])
submission = submission.loc[:, ['ID', 'item_cnt_month']]
print(submission.shape)
submission.head()

(214200, 2)


,ID,item_cnt_month
0,0,1.118458
1,1,1.414916
2,2,1.593280
3,3,2.004942
4,4,1.414916


In [82]:
# Export submission to .csv file and send to Kaggle
submission.to_csv('thai_an_le_submission.csv', index = False)

### Useful resources and references:

- https://www.kaggle.com/code/thnhnguyntrngtt/predict-future-sales-0-86-solution (partly written in Vietnamese - LGBM with early stopping)

- https://www.kaggle.com/code/cngnguynt04/lightgbm-0-87442 (contains many useful tips to optimize performance)